# COMPASS multivariate models: adjust-only covariates

Re-runs every prediction from `03_multivariate.ipynb` (elastic-net Cox and
XGBoost, both/baseline, the available-case sensitivity, the clinical
combinations, risk stratification) plus both predictive-checklist configs with
`--adjust-only-mode drop`.

**Adjust-only columns** are the data-density/batch covariates rather than biology:
every `LAB__n_observations`, every `__missing` indicator, and the static
covariates (`PANEL_VERSION_*` dummies, `SOMATIC_N_GENES_ALTERED`). Age stays a
reported predictor. They are used in training and then removed from the
deployed model:

| Model | In training | In the final model |
|---|---|---|
| Elastic-net Cox | fit jointly with the features (n_obs/missing penalized, static unpenalized) | zeroed (= training mean) in every score; dropped from `cox_agg_multivariable.csv`, audited in `cox_agg_multivariable_adjusters.csv` |
| XGBoost | adjuster-only Coxnet linear predictor used as `base_margin` (offset) | trees never see them; predictions use a zero offset |
| Checklists | not candidate items; enter the calibrator as `logit p = a + b·score + Z·γ` | probabilities evaluated at Z = 0; γ stored for audit only |

Every score (tuning-fold C-index, out-of-fold risks, test metrics, risk files)
is the features-only score. Outputs go to a parallel tree
(`local_runs_<label>_adjust_only`, `predictive_checklists/*_adjust_only`); the
prediction inputs are shared with 03 and the default-mode outputs are never
touched. The checklist comparison baselines are restricted to the same candidate
pool (no n_obs/missing) but are not themselves adjusted.

In [ ]:
ARMS = ["adt"]
ENDPOINTS = ("platinum",)
COHORTS = ("all",)
EXCLUSIONS = ("none",)
ADJUST_ONLY_MODE = "drop"
OVERWRITE = False  # True: refit and replace existing outputs; False: resume/skip

import sys
sys.path.insert(0, ".")
import compass_pipeline as cp

cp.N_FOLDS = 5
cp.FORCE_RERUN = OVERWRITE
RUNS = cp.make_endpoint_runs(
    ARMS, endpoints=ENDPOINTS, cohorts=COHORTS, exclusions=EXCLUSIONS,
    adjust_only_mode=ADJUST_ONLY_MODE,
)
# The matching default-mode runs from 03, for the comparison cells only.
DEFAULT_RUNS = cp.make_endpoint_runs(ARMS, endpoints=ENDPOINTS, cohorts=COHORTS, exclusions=EXCLUSIONS)
print(f"{len(RUNS)} run(s): {[r['label'] for r in RUNS]}")
print(f"landmarks: {sorted({lm for r in RUNS for lm in r['landmarks']})}")
print(f"models: {sorted({(m, c) for m, c, _ in cp.MULTIVARIATE_TASK_SPECS})}")

## Run multivariate models

Elastic-net (both/baseline) and XGBoost (both/baseline). In drop mode the
baseline arm keeps age as its only score term (the static covariates are fit
but zeroed).

In [ ]:
for run in RUNS:
    cp.run_multivariate(run)

## Available-case Gleason sensitivity

In [ ]:
for run in RUNS:
    cp.run_multivariate_available_case_sensitivity(run)

## Gleason / somatic / labs component combinations

In [ ]:
for run in RUNS:
    cp.run_multivariate_clinical_combinations(run)

## Risk-score stratification

Reads the drop-mode risk files from the parallel tree; `test` and `cv_oof`
are kept in separate subdirectories as in 03.

In [ ]:
for run in RUNS:
    cp.run_risk_stratification(run)

## Predictive checklists

Both configs, run as subprocesses exactly as in
`predictive_checklists/run_predictive_checklists*.ipynb` but with
`--adjust-only-mode drop` and `_adjust_only` output directories. Preflight is
mode-independent, so it is not repeated here. Gated by `RUN_CHECKLISTS`
(off by default, like `RUN_TRAINING` in those notebooks).

In [ ]:
from pathlib import Path
import subprocess

RUN_CHECKLISTS = False
CHECKLIST_OVERWRITE = False  # archives previous generated results
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "survival_common").is_dir())
CHECKLIST_SUBPROJECT = REPO_ROOT / "COMPASS/survival_analysis/predictive_checklists"
INPUTS_ROOT = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis")
CHECKLIST_ROOT = INPUTS_ROOT / "predictive_checklists"
GLEASON_TRIO_SUBSET = "somatic_gleason/gleason_somatic_available_case"

CHECKLIST_RUNS = {
    # name: (config, input args, landmarks, horizons, default output, adjust-only output)
    "full_adt": (
        CHECKLIST_SUBPROJECT / "default_config.json",
        ["--inputs-root", str(INPUTS_ROOT)],
        [0, 90, 180], [365, 730, 1095, 1460, 1825],
        CHECKLIST_ROOT / "full_adt", CHECKLIST_ROOT / "full_adt_adjust_only",
    ),
    "gleason_trio": (
        CHECKLIST_SUBPROJECT / "gleason_trio_config.json",
        ["--platinum-inputs", str(INPUTS_ROOT / "prediction_inputs_adt" / GLEASON_TRIO_SUBSET),
         "--nepc-inputs", str(INPUTS_ROOT / "prediction_inputs_adt_nepc" / GLEASON_TRIO_SUBSET)],
        [0, 90, 180], [1825],
        CHECKLIST_ROOT / "gleason_trio_labs", CHECKLIST_ROOT / "gleason_trio_labs_adjust_only",
    ),
}

if RUN_CHECKLISTS:
    for name, (config, input_args, landmarks, horizons, _, output) in CHECKLIST_RUNS.items():
        cmd = [sys.executable, "-m", "COMPASS.survival_analysis.predictive_checklists",
               "--config", str(config), *input_args,
               "--landmarks", *map(str, landmarks), "--horizons", *map(str, horizons),
               "--adjust-only-mode", ADJUST_ONLY_MODE, "--output-dir", str(output)]
        cmd += ["--overwrite"] if CHECKLIST_OVERWRITE else []
        print(f"[run ] checklist {name} -> {output}")
        subprocess.run(cmd, cwd=REPO_ROOT, check=True)

## Default vs adjust-only comparison

Held-out metrics per (model, landmark, config, endpoint) in each mode. A drop
in discrimination is expected where n_observations/missingness carried real
signal: that signal is now attributed to the adjusters and removed from the
deployed score. Rows missing in either mode show `NaN`.

In [ ]:
import pandas as pd

KEYS = ["model", "landmark", "config", "endpoint"]
METRICS = ["c_index", "mean_auc_t", "integrated_brier"]

def _ok(frame):
    return frame.loc[frame["status"] == "ok", KEYS + METRICS]

comparison_dfs = {}
for default_run, drop_run in zip(DEFAULT_RUNS, RUNS):
    assert cp.run_key(default_run) == cp.run_key(drop_run)
    merged = _ok(cp.summarize_outputs(default_run)).merge(
        _ok(cp.summarize_outputs(drop_run)), on=KEYS, how="outer", suffixes=("_default", "_adjust_only"),
    )
    for metric in METRICS:
        merged[f"delta_{metric}"] = merged[f"{metric}_adjust_only"] - merged[f"{metric}_default"]
    comparison_dfs[cp.run_key(drop_run)] = merged.sort_values(KEYS).reset_index(drop=True)
    print(f"=== {cp.run_key(drop_run)} ===")
    display(comparison_dfs[cp.run_key(drop_run)])

In [ ]:
checklist_rows = []
for name, (*_, default_output, drop_output) in CHECKLIST_RUNS.items():
    for mode, output in (("default", default_output), ("adjust_only", drop_output)):
        path = output / "metrics.csv"
        if not path.exists():
            continue
        frame = pd.read_csv(path)
        frame["checklist_run"] = name
        frame["mode"] = mode
        checklist_rows.append(frame)

if checklist_rows:
    checklist_keys = [c for c in ["checklist_run", "endpoint", "landmark_days", "population",
                                  "horizon_days", "model"] if all(c in f for f in checklist_rows)]
    checklist_metrics = pd.concat(checklist_rows, ignore_index=True)
    checklist_comparison = (
        checklist_metrics.loc[checklist_metrics["status"] == "ok"]
        .pivot_table(index=checklist_keys, columns="mode", values=["auc", "brier", "n_items"])
    )
    checklist_comparison.columns = [f"{metric}_{mode}" for metric, mode in checklist_comparison.columns]
    for metric in ("auc", "brier", "n_items"):
        if {f"{metric}_default", f"{metric}_adjust_only"} <= set(checklist_comparison.columns):
            checklist_comparison[f"delta_{metric}"] = (
                checklist_comparison[f"{metric}_adjust_only"] - checklist_comparison[f"{metric}_default"]
            )
    display(checklist_comparison.reset_index())
else:
    print("No checklist metrics.csv found in either mode.")

## Sanity check: no adjusters in the final models

Top non-zero Cox coefficients and top XGBoost gain features in drop mode,
plus the adjuster audit table. Any adjuster name in the first two is a bug.

In [ ]:
ADJUSTER_SUFFIXES = ("__n_observations", "__missing")
ADJUSTER_PREFIXES = ("PANEL_VERSION_", "SOMATIC_N_GENES_ALTERED")

def is_adjuster(name):
    return str(name).endswith(ADJUSTER_SUFFIXES) or str(name).startswith(ADJUSTER_PREFIXES)

TOP_N = 15
for run in RUNS:
    for landmark in run["landmarks"]:
        cox_dir = run["output_dir"] / "cox" / f"landmark_{landmark}" / "both"
        xgb_dir = run["output_dir"] / "xgboost" / f"landmark_{landmark}" / "both"
        coef_path = cox_dir / "cox_agg_multivariable.csv"
        audit_path = cox_dir / "cox_agg_multivariable_adjusters.csv"
        importance_path = xgb_dir / "landmark_xgboost_feature_importance.csv"
        print(f"=== {run['label']} +{landmark}d ===")
        if coef_path.exists():
            coefs = pd.read_csv(coef_path)
            leaked = coefs.loc[coefs["feature"].map(is_adjuster), "feature"].tolist()
            print(f"Cox: {len(coefs)} reported terms, adjusters leaked: {leaked or 'none'}")
            display(coefs.loc[coefs["coef"] != 0, ["feature", "coef", "exp(coef)"]].head(TOP_N))
        else:
            print(f"Cox: missing {coef_path}")
        if audit_path.exists():
            audit = pd.read_csv(audit_path)
            print(f"Cox adjuster audit: {len(audit)} terms, {(audit['coef'] != 0).sum()} non-zero")
        if importance_path.exists():
            importance = pd.read_csv(importance_path)
            leaked = importance.loc[importance["feature"].map(is_adjuster), "feature"].unique().tolist()
            print(f"XGBoost: adjusters leaked: {leaked or 'none'}")
            display(importance.loc[importance["gain"] > 0, ["feature", "gain"]].head(TOP_N))
        else:
            print(f"XGBoost: missing {importance_path}")

for name, (*_, drop_output) in CHECKLIST_RUNS.items():
    for checklist in sorted(drop_output.glob("*/landmark_*/*/horizon_*/checklist.txt")):
        text = checklist.read_text()
        items = [line for line in text.splitlines() if line.startswith("[ ]")]
        leaked = [line for line in items if "__n_observations" in line or line.endswith(" is missing")]
        flag = f"  <-- adjuster items: {leaked}" if leaked else ""
        print(f"--- {name}/{checklist.relative_to(drop_output)}{flag}")
        print(text)